# 17 · Population cells (Part D, step D2)

**What this does:** turns the WorldPop 2026 100 m population grid of India into a table with **one row per populated
cell** — the "people" side of every access measure — and tags each cell with its state, district, sub-district and
(in Kerala) local body.

**Input:** WorldPop R2025A 2026, India, 100 m constrained (total population and the age bands 5–9, 10–14, 15–19;
CC BY 4.0, DOI 10.5258/SOTON/WP00841), all on one grid of 35,040 × 34,507 ≈ **1.21 billion cells**; area polygons
from the silver lake (notebook 16).

**How (Spark, map phase without shuffle):** the grid is cut into 1,024 × 1,024 blocks; the blocks are spread over the
Spark tasks; each task reads its blocks with rasterio, keeps cells with population > 0 and burns the area polygons
onto the same block (`rasterio.features.rasterize`: a cell belongs to the polygon containing its **centre** — the
same rule used for the Kerala clips in notebook 11). The worker code is `scripts/bigdata/population_cells.py`.

**Cells whose centre is in no polygon** (coastline drawn coarser than the grid, gaps between polygons) get the
**nearest** polygon within ~2 km (local bodies ~0.5 km) through a Sedona distance join; cells farther away keep
code 0 and are reported, never dropped.

**Output:** `data/lake/silver/population_cells/` (Parquet, zstd, partitioned by state): `cell_id` (= row × 35,040 +
col; the centre's lon/lat come back with `population_cells.lonlat_sql`), `pop`, `pop_5_9`, `pop_10_14`, `pop_15_19`,
area codes, `tag_nearest`, `age_nodata`; and `area_units.parquet` (code → id, name). No geometry column is stored
(it would add ~3 GB); later steps build the point with `ST_Point(lon, lat)`.

**Checks:** Kerala is run first; cell counts and population sums must equal a direct rasterio read; the Kerala
total must equal the Kerala clip; every cell appears once; the tagging is compared with a Sedona point-in-polygon
join on all Kerala cells and on a random 1 million India cells.

## Project folder (laptop or Colab)

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))

In [2]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio", "rasterio"], check=True)

## Install the Part D packages

In [3]:
# Part D packages, pinned in requirements-bigdata.txt (PySpark 3.5.9, apache-sedona 1.9.1, pyarrow).
# Colab also needs Java 17, which Spark runs on (the laptop already has it: JAVA_HOME).
import os
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements-bigdata.txt")], check=True)
if IN_COLAB:
    subprocess.run("apt-get install -y -qq openjdk-17-jdk-headless > /dev/null", shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

## Start Spark + Sedona, check disk space

In [4]:
import random
import shutil
import time

import numpy as np
import pandas as pd
import rasterio
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import BinaryType, IntegerType, StringType, StructField, StructType
from bigdata.spark_session import get_spark

sys.path.insert(0, str(PROJECT / "scripts" / "bigdata"))
import population_cells as pc  # the worker code (same module the Spark tasks import)

RASTERS = PROJECT / "data" / "raw" / "worldpop"
SILVER = PROJECT / "data" / "lake" / "silver"
OUT = SILVER / "population_cells"
TMP = (Path("/content/tmp") if IN_COLAB else PROJECT / "data" / "tmp") / "population_cells_raw"
free_gb = shutil.disk_usage(PROJECT).free / 1e9
print(f"free disk: {free_gb:.1f} GB")
assert free_gb > 5, "less than 5 GB free: stop"

# 6 cores, not 12: every core runs a Python worker holding the boundary polygons (~0.5 GB each); with 12 the
# laptop ran out of memory (other programs use ~9 GB of its 16 GB). Failed tasks are retried.
spark = get_spark(PROJECT, cores=6, memory="6g")
spark.sparkContext.addPyFile(str(PROJECT / "scripts" / "bigdata" / "population_cells.py"))  # ship the worker code
WIDTH, HEIGHT, TRANSFORM = pc.grid(RASTERS)
LON, LAT = pc.lonlat_sql(RASTERS)
print(f"WorldPop grid: {WIDTH:,} x {HEIGHT:,} = {WIDTH * HEIGHT / 1e9:.2f} billion cells | blocks of {pc.BLOCK}: {len(pc.blocks(RASTERS)):,}")
print("cores:", spark.sparkContext.defaultParallelism, "| Spark UI:", spark.sparkContext.uiWebUrl)

free disk: 14.1 GB


WorldPop grid: 35,040 x 34,507 = 1.21 billion cells | blocks of 1024: 1,190
cores: 6 | Spark UI: http://127.0.0.1:4040


## Area polygons and their codes

In [5]:
# Area polygons with the same integer codes the workers use (pc.load_units), as Sedona tables for the checks
unit_tables, lookup = {}, []
schema = StructType([StructField("code", IntegerType()), StructField("unit_id", StringType()),
                     StructField("unit_name", StringType()), StructField("wkb", BinaryType())])
for code_col, (table, _, _) in pc.UNITS.items():
    g = pc.load_units(SILVER, code_col)
    pdf = pd.DataFrame({"code": g["code"], "unit_id": g["unit_id"].astype(str), "unit_name": g["unit_name"],
                        "wkb": g.geometry.to_wkb()})
    unit_tables[code_col] = spark.createDataFrame(pdf, schema).selectExpr("code", "unit_id", "unit_name",
                                                                          "ST_GeomFromWKB(wkb) AS geometry").cache()
    lookup.append(pdf.drop(columns="wkb").assign(layer=table))
    print(f"{code_col:17s} {table:20s} {len(g):5d} polygons")
lookup = pd.concat(lookup)[["layer", "code", "unit_id", "unit_name"]]
KERALA = int(lookup.query("layer == 'states' and unit_name == 'Kerala'")["code"].iloc[0])

state_code        states                  36 polygons


district_code     districts              735 polygons


subdistrict_code  subdistricts          6824 polygons
local_body_code   kerala_local_bodies   1027 polygons


## The Spark job and the checks

In [6]:
def cells_job(block_list, partitions):
    """The map phase: each Spark task turns its raster blocks into populated, tagged cells (no shuffle)."""
    raster, silver = str(RASTERS), str(SILVER)

    def run(batches):
        import population_cells
        yield from population_cells.map_blocks(batches, raster, silver)

    random.Random(0).shuffle(block_list)  # mix sea / desert / city blocks so every task gets a similar load
    blocks_df = spark.sparkContext.parallelize(block_list, partitions).toDF("row0 int, col0 int, height int, width int")
    return blocks_df.mapInPandas(run, pc.SCHEMA)


def raster_totals(rows=None, cols=None):
    """Independent check without Spark: count and sums of populated cells read directly with rasterio."""
    r_lo, r_hi = rows or (0, HEIGHT)
    c_lo, c_hi = cols or (0, WIDTH)
    tot = {"cells": 0}
    for name, f in pc.BANDS.items():
        s = 0.0
        with rasterio.open(RASTERS / f) as r, rasterio.open(RASTERS / pc.BANDS["pop"]) as p:
            for r0 in range(r_lo, r_hi, 1024):
                win = ((r0, min(r0 + 1024, r_hi)), (c_lo, c_hi))
                keep = p.read(1, window=win) > 0
                a = r.read(1, window=win)[keep]
                s += float(a[a >= 0].sum(dtype="float64"))
                if name == "pop":
                    tot["cells"] += int(keep.sum())
        tot[name] = s
    return tot


def totals(df):
    return df.agg(F.count("*").alias("cells"), *[F.sum(c).alias(c) for c in pc.BANDS]).first().asDict()


def sedona_check(cells, code_col, label):
    """Same cells, tagged again by a Sedona point-in-polygon join (polygons broadcast to every task)."""
    t = time.time()
    pts = cells.selectExpr("cell_id", code_col, f"ST_Point({LON}, {LAT}) AS geom")
    hits = pts.alias("c").join(F.broadcast(unit_tables[code_col]).alias("u"), F.expr("ST_Intersects(u.geometry, c.geom)")) \
              .groupBy("c.cell_id").agg(F.collect_set("u.code").alias("codes"))
    j = pts.join(hits, "cell_id", "left")
    r = j.agg(F.count("*").alias("cells"),
              F.sum(F.when(F.expr(f"array_contains(codes, {code_col}) OR ({code_col} = 0 AND codes IS NULL)"), 1).otherwise(0)).alias("agree"),
              F.sum(F.when(F.size("codes") > 1, 1).otherwise(0)).alias("in_two_polygons")).first()
    print(f"{label}: {r['cells']:,} cells | same unit as Sedona: {r['agree'] / r['cells']:.4%} | "
          f"in overlapping polygons: {r['in_two_polygons']:,} | {time.time() - t:.0f} s")
    return r

## Kerala test run

In [7]:
# Kerala test first: only the blocks of the Kerala raster window (same grid as notebook 11's Kerala clips)
with rasterio.open(RASTERS / "kerala" / "kerala_pop_2026_CN_100m_R2025A_v1.tif") as k:
    col0, row0 = [round(v) for v in ~TRANSFORM * (k.transform.c, k.transform.f)]
    k_rows, k_cols = (row0, row0 + k.height), (col0, col0 + k.width)
    a = k.read(1)
    kerala_clip_pop = float(a[a > 0].sum(dtype="float64"))
t = time.time()
kerala_cells = cells_job(pc.blocks(RASTERS, k_rows, k_cols), 24).cache()
kt = totals(kerala_cells)
print(f"Kerala window: {kt['cells']:,} populated cells in {time.time() - t:.0f} s")
ind = raster_totals(k_rows, k_cols)
for c in ["cells", *pc.BANDS]:
    assert abs(kt[c] - ind[c]) <= (0 if c == "cells" else 1e-6 * ind[c]), f"{c}: Spark {kt[c]} vs rasterio {ind[c]}"
in_kerala = kerala_cells.filter(F.col("state_code") == KERALA).agg(F.sum("pop")).first()[0]
print(f"same totals as reading the rasters directly | population with centre in Kerala: {in_kerala / 1e6:.3f} M "
      f"| Kerala clip (notebook 11): {kerala_clip_pop / 1e6:.3f} M")
assert abs(in_kerala - kerala_clip_pop) <= 1e-4 * kerala_clip_pop, "Kerala total differs from the clip"

Kerala window: 5,240,727 populated cells in 46 s


same totals as reading the rasters directly | population with centre in Kerala: 37.418 M | Kerala clip (notebook 11): 37.418 M


### Kerala: compare with Sedona point-in-polygon

In [8]:
# Check the tagging against Sedona on every Kerala cell (local bodies, sub-districts)
kc = kerala_cells.filter(F.col("state_code") == KERALA)
sedona_check(kc, "local_body_code", "Kerala local bodies")
sedona_check(kc, "subdistrict_code", "Kerala sub-districts")
kerala_cells.unpersist()

Kerala local bodies: 2,972,121 cells | same unit as Sedona: 100.0000% | in overlapping polygons: 2 | 21 s


Kerala sub-districts: 2,972,121 cells | same unit as Sedona: 100.0000% | in overlapping polygons: 0 | 21 s


DataFrame[cell_id: bigint, pop: float, pop_5_9: float, pop_10_14: float, pop_15_19: float, state_code: int, district_code: int, subdistrict_code: int, local_body_code: int, age_nodata: boolean]

## All India run

In [9]:
# All India: every block of the grid -> populated cells, written once to a temporary Parquet folder
t = time.time()
cells_job(pc.blocks(RASTERS), 120).write.mode("overwrite").parquet(str(TMP))
raw = spark.read.parquet(str(TMP))
it = totals(raw)
map_s = time.time() - t
print(f"India: {it['cells']:,} populated cells | {it['pop'] / 1e6:,.2f} M people | map phase {map_s / 60:.1f} min")
t = time.time()
ind = raster_totals()
print(f"direct rasterio read for the check: {time.time() - t:.0f} s")
for c in ["cells", *pc.BANDS]:
    assert abs(it[c] - ind[c]) <= (0 if c == "cells" else 1e-6 * ind[c]), f"{c}: Spark {it[c]} vs rasterio {ind[c]}"
print("Spark totals = raster totals:", {c: round(ind[c] / 1e6, 2) for c in pc.BANDS}, "million")

India: 140,086,378 populated cells | 1,470.30 M people | map phase 4.1 min


direct rasterio read for the check: 71 s
Spark totals = raster totals: {'pop': 1470.3, 'pop_5_9': 117.15, 'pop_10_14': 122.58, 'pop_15_19': 126.42} million


## Cells outside every polygon: nearest unit (Sedona)

In [10]:
# Cells whose centre is in no polygon (coastline drawn coarser than the raster, gaps between polygons):
# give them the nearest polygon within a short distance (Sedona distance join); farther cells stay 0 and are reported.
MAX_DEG = {"state_code": 0.02, "district_code": 0.02, "subdistrict_code": 0.02, "local_body_code": 0.005}  # ~2 km / ~0.5 km


def missing(df):
    return (F.col("state_code") == 0) | (F.col("district_code") == 0) | (F.col("subdistrict_code") == 0) | \
           ((F.col("state_code") == KERALA) & (F.col("local_body_code") == 0))


def nearest(df, code_col, todo):
    pts = df.filter(todo).selectExpr("cell_id", f"ST_Point({LON}, {LAT}) AS geom")
    pairs = pts.alias("c").join(F.broadcast(unit_tables[code_col]).alias("u"),
                                F.expr(f"ST_DWithin(u.geometry, c.geom, {MAX_DEG[code_col]})")) \
               .select("c.cell_id", "u.code", F.expr("ST_Distance(u.geometry, c.geom)").alias("d"))
    best = pairs.withColumn("rk", F.row_number().over(Window.partitionBy("cell_id").orderBy("d", "code"))) \
                .filter("rk = 1").select("cell_id", F.col("code").alias("new_code"))
    return df.join(best, "cell_id", "left") \
             .withColumn(code_col, F.when(F.col(code_col) == 0, F.coalesce("new_code", F.lit(0))).otherwise(F.col(code_col))) \
             .drop("new_code")


t = time.time()
untagged = raw.filter(missing(raw)).cache()
before = untagged.agg(F.count("*"), F.sum("pop")).first()
print(f"cells missing at least one area: {before[0]:,} ({before[1] / 1e6:.3f} M people)")
fixed = untagged
for code_col in ["state_code", "district_code", "subdistrict_code"]:
    fixed = nearest(fixed, code_col, F.col(code_col) == 0)
fixed = nearest(fixed, "local_body_code", (F.col("state_code") == KERALA) & (F.col("local_body_code") == 0))
fixed = fixed.withColumn("tag_nearest", F.lit(True)).cache()
report = fixed.select([F.sum(F.when(F.col(c) == 0, F.col("pop")).otherwise(0)).alias(c) for c in ["state_code", "district_code", "subdistrict_code"]] +
                      [F.sum(F.when((F.col("state_code") == KERALA) & (F.col("local_body_code") == 0), F.col("pop")).otherwise(0)).alias("local_body_code")]).first()
print("people still without a unit after the nearest step (million):", {k: round(v / 1e6, 4) for k, v in report.asDict().items()},
      f"| {time.time() - t:.0f} s")

cells missing at least one area: 183,805 (1.677 M people)


people still without a unit after the nearest step (million): {'state_code': 0.0932, 'district_code': 0.0268, 'subdistrict_code': 0.0204, 'local_body_code': 0.2256} | 942 s


## Write the silver table

In [11]:
# Final silver table: tagged cells + fixed cells, partitioned by state (state names joined from a 36-row table)
t = time.time()
names = spark.createDataFrame(lookup.query("layer == 'states'")[["code", "unit_name"]]
                              .rename(columns={"code": "state_code", "unit_name": "state"}))
cells = raw.filter(~missing(raw)).withColumn("tag_nearest", F.lit(False)).unionByName(fixed)
cells = cells.join(F.broadcast(names), "state_code", "left").fillna({"state": "(outside boundaries)"})
cells.write.mode("overwrite").option("compression", "zstd").partitionBy("state").parquet(str(OUT))
lookup.to_parquet(SILVER / "area_units.parquet", index=False)
print(f"written {OUT} in {(time.time() - t) / 60:.1f} min")
untagged.unpersist(); fixed.unpersist()
shutil.rmtree(TMP, ignore_errors=True)

written C:\Users\ASUS\Desktop\sem3\BIG SPATIAL DATA\PROJECT- SPATIAL INEQUALITY ASESSMENT\data\lake\silver\population_cells in 4.2 min


## Final checks and population by state

In [12]:
final = spark.read.parquet(str(OUT))
ft = totals(final)
for c in ["cells", *pc.BANDS]:
    assert abs(ft[c] - ind[c]) <= (0 if c == "cells" else 1e-6 * ind[c]), f"{c}: final {ft[c]} vs raster {ind[c]}"
assert final.select("cell_id").distinct().count() == ft["cells"], "duplicate cells"
files = list(OUT.rglob("*.parquet"))
size_mb = sum(f.stat().st_size for f in files) / 1e6
print(f"final table: {ft['cells']:,} cells, every cell once, totals = raster | {len(files)} files, {size_mb:,.0f} MB")
by_state = final.groupBy("state").agg(F.count("*").alias("cells"), (F.sum("pop") / 1e6).alias("pop_million"),
                                      F.sum(F.col("tag_nearest").cast("int")).alias("cells_by_nearest"),
                                      F.sum(F.col("age_nodata").cast("int")).alias("cells_age_nodata")) \
                 .orderBy(F.desc("pop_million")).toPandas()
print(by_state.round(3).to_string(index=False))

final table: 140,086,378 cells, every cell once, totals = raster | 6210 files, 2,114 MB


                                   state    cells  pop_million  cells_by_nearest  cells_age_nodata
                           Uttar Pradesh 13754598      252.162               951                 0
                                   Bihār  5642880      135.357              3297                 0
                             Mahārāshtra 13878319      133.559              6940                 0
                             West Bengal  5662472      104.089             53011                 0
                          Madhya Pradesh 11528051       92.068                 4                 0
                               Rājasthān 16890420       85.964              3242                 0
                              Tamil Nādu  8055364       80.608              5158                 0
                                 Gujarāt 10071419       75.373             11769                 0
                               Karnātaka  8701420       71.342              2094                 0
          

## Sedona check on 1 million random India cells

In [13]:
# Sedona check on a random sample of 1 million India cells (sub-districts and districts)
sample = final.sample(fraction=1e6 / ft["cells"], seed=7).cache()
sedona_check(sample, "subdistrict_code", "India sample, sub-districts")
sedona_check(sample, "district_code", "India sample, districts")
sample.unpersist()

India sample, sub-districts: 998,085 cells | same unit as Sedona: 99.9504% | in overlapping polygons: 0 | 31 s


India sample, districts: 998,085 cells | same unit as Sedona: 99.9634% | in overlapping polygons: 0 | 18 s


DataFrame[state_code: int, cell_id: bigint, pop: float, pop_5_9: float, pop_10_14: float, pop_15_19: float, district_code: int, subdistrict_code: int, local_body_code: int, age_nodata: boolean, tag_nearest: boolean, state: string]

## Catalog

In [14]:
cat = pd.read_csv(SILVER / "_catalog.csv")
cat = pd.concat([cat[cat["table"] != "population_cells"], pd.DataFrame([{
    "table": "population_cells", "rows": ft["cells"], "with_geometry": 0, "invalid_geometry": 0, "files": len(files),
    "size_mb": round(size_mb, 1), "partitioned_by": "state",
    "source": "data/raw/worldpop/ind_*_2026_CN_100m_R2025A_v1.tif (WorldPop R2025A) + silver boundaries; "
              "cell centre from cell_id (population_cells.lonlat_sql)"}])])
cat.to_csv(SILVER / "_catalog.csv", index=False)
print(cat[["table", "rows", "files", "size_mb", "partitioned_by"]].to_string(index=False))
print(f"map phase {map_s / 60:.1f} min | free disk now {shutil.disk_usage(PROJECT).free / 1e9:.1f} GB")
spark.stop()

              table      rows  files  size_mb partitioned_by
  health_facilities    228986      1     23.6            NaN
      schools_india   1070017     35    125.4     state_name
     schools_kerala     11764      1      2.8            NaN
      fire_stations       129      1      0.0            NaN
             states        36      1     16.9            NaN
          districts       735      1     29.6            NaN
       subdistricts      6824      1     38.5            NaN
kerala_local_bodies      1027      1      3.3            NaN
   population_cells 140086378   6210   2114.3          state
map phase 4.1 min | free disk now 14.2 GB
